# 03 — Memory & State (Checkpointers)

**Deprecated:** `ConversationBufferMemory`, `ConversationSummaryMemory`, and friends (from `langchain.memory`) — moved to `langchain-classic`, no longer the recommended pattern.

**Current:** memory is handled by a **checkpointer** — you attach one to your agent, and it persists conversation state per `thread_id`. This is a LangGraph concept that `create_agent` exposes directly, so what you learn here also applies if you move to raw LangGraph later.


In [ ]:
import os
from getpass import getpass
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
MODEL_ID = "openai:gpt-4.1-mini"
model = init_chat_model(MODEL_ID, temperature=0)

@tool
def add_to_watchlist(ticker: str) -> str:
    """Add a stock ticker to the user's watchlist."""
    return f"Added {ticker.upper()} to your watchlist."

tools = [add_to_watchlist]

## 1. Without memory — each call is stateless

If you don't attach a checkpointer, the agent has no idea what happened in a previous `.invoke()` call.


In [ ]:
stateless_agent = create_agent(model=model, tools=tools)

r1 = stateless_agent.invoke({"messages": [{"role": "user", "content": "Add TCS to my watchlist."}]})
print(r1["messages"][-1].content)

r2 = stateless_agent.invoke({"messages": [{"role": "user", "content": "What did I just ask you to do?"}]})
print(r2["messages"][-1].content)  # It won't know — no shared state between calls

## 2. With memory — attach an `InMemorySaver` checkpointer

`InMemorySaver` is a simple in-process checkpointer, good for notebooks/dev. For production you'd use a persistent one (e.g. backed by Postgres or SQLite — LangGraph ships integrations for these).

The key idea: every `.invoke()` needs a `thread_id` in its config. All calls with the same `thread_id` share conversation history; different `thread_id`s are fully isolated (think: different chat sessions or different users).


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

memory_agent = create_agent(model=model, tools=tools, checkpointer=checkpointer)

config = {"configurable": {"thread_id": "ankur-session-1"}}

r1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Add TCS to my watchlist."}]},
    config=config,
)
print(r1["messages"][-1].content)

r2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What did I just ask you to do?"}]},
    config=config,  # same thread_id -> same conversation history
)
print(r2["messages"][-1].content)

## 3. Isolating separate conversations with different `thread_id`s

This is how you'd support multiple users or multiple independent chat sessions in the same app.


In [ ]:
other_config = {"configurable": {"thread_id": "different-session"}}

r3 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What did I just ask you to do?"}]},
    config=other_config,  # different thread_id -> no shared history with session 1
)
print(r3["messages"][-1].content)  # should have no memory of the TCS request

## 4. Inspecting stored state directly

You can pull the full checkpointed state for a thread without invoking the agent — useful for debugging or building a "conversation history" UI.


In [ ]:
state = memory_agent.get_state(config)
for m in state.values["messages"]:
    print(f"[{m.__class__.__name__}] {m.content}")

## 5. Trimming history for long conversations

Left unchecked, message history grows forever and eats your context window / cost. A common current pattern is a **middleware** that trims older messages before they hit the model. (Middleware is a v1.0 concept — a way to hook into the agent's execution without rewriting the whole graph.)


In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

# This automatically summarizes older parts of the conversation once the token count
# for the message history crosses `max_tokens_before_summary`, keeping recent turns verbatim.
trimming_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
    middleware=[
        SummarizationMiddleware(
            model=model,
            max_tokens_before_summary=2000,
            messages_to_keep=6,
        )
    ],
)
print("Agent with automatic history summarization ready.")

---
### Key takeaways
- `ConversationBufferMemory` is deprecated — use a **checkpointer** instead.
- `InMemorySaver()` for dev/notebooks; swap for a persistent checkpointer (Postgres/SQLite) in production.
- Every stateful call needs a `config={"configurable": {"thread_id": "..."}}` — same `thread_id` = shared memory.
- `agent.get_state(config)` lets you inspect stored history directly.
- `SummarizationMiddleware` keeps long conversations from blowing up your context window/cost.

**Next:** `04_structured_output.ipynb` — getting reliable, typed data back from the model instead of free text.
